In [ ]:
!pip install torch_geometric

# READ DATASET

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.datasets.movie_lens_1m import MovieLens1M

torch.manual_seed(42)
np.random.seed(42)


dataset = MovieLens1M(root="./data/MovieLens1M")
data = dataset[0]

num_users_raw = data["user"].num_nodes
num_movies_raw = data["movie"].num_nodes

edge_index = data["user", "rates", "movie"].edge_index
ratings = data["user", "rates", "movie"].rating.float()
timestamps = data["user", "rates", "movie"].time.long()

print("=" * 60)
print("RAW DATASET")
print("=" * 60)
print(f"Users        : {num_users_raw}")
print(f"Movies       : {num_movies_raw}")
print(f"Interactions : {edge_index.size(1)}")

In [ ]:
df = pd.DataFrame({
    "user": edge_index[0].numpy(),
    "movie": edge_index[1].numpy(),
    "rating": ratings.numpy(),
    "timestamp": timestamps.numpy()
})


print("\nRating Distribution")
print(df["rating"].value_counts().sort_index())

user_interactions = df.groupby("user").size()
movie_interactions = df.groupby("movie").size()

print("\nInteraction Statistics")
print(f"User   : min={user_interactions.min()} "
      f"median={user_interactions.median()} "
      f"max={user_interactions.max()}")

print(f"Movie  : min={movie_interactions.min()} "
      f"median={movie_interactions.median()} "
      f"max={movie_interactions.max()}")

# Convert Explicit -> Implicit Feedback

In [ ]:
POS_THRESHOLD = 4.0

df = df[df["rating"] >= POS_THRESHOLD].copy()

print("\nImplicit Feedback")
print(f"Positive interactions: {len(df)}")

# Remove users with fewer than 5 positive interactions

In [ ]:
MIN_INTERACTIONS = 5

user_counts = df.groupby("user").size()

valid_users = user_counts[user_counts >= MIN_INTERACTIONS].index

df = df[df["user"].isin(valid_users)].copy()

print(f"Remaining users : {df['user'].nunique()}")
print(f"Remaining movies: {df['movie'].nunique()}")

# Re-index User and Movie IDs

In [ ]:
unique_users = np.sort(df["user"].unique())
unique_movies = np.sort(df["movie"].unique())

user2idx = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}

df["user_idx"] = df["user"].map(user2idx)
df["movie_idx"] = df["movie"].map(movie2idx)

num_users = len(unique_users)
num_items = len(unique_movies)

print("\nAfter Re-index")
print(f"Users : {num_users}")
print(f"Items : {num_items}")

# SPLIT TRAIN/VAL/TEST

In [ ]:
df = df.sort_values(["user_idx", "timestamp"]).copy()
df["split"] = "train"

last_two = df.groupby("user_idx", group_keys=False).tail(2)
val_idx = last_two.groupby("user_idx", group_keys=False).head(1).index
test_idx = last_two.groupby("user_idx", group_keys=False).tail(1).index

df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"
df = df.reset_index(drop=True)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("\nDataset Split")
print(df["split"].value_counts())

In [ ]:
train_edges = torch.tensor(
    train_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

val_edges = torch.tensor(
    val_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

test_edges = torch.tensor(
    test_df[["user_idx", "movie_idx"]].values,
    dtype=torch.long
)

In [ ]:
# =========================================================
# Evaluation lookup structures
# =========================================================
# train_user_pos_items[u] is the set of remapped item IDs that user u
# interacted with in the TRAIN split only. Validation/test items are excluded.
train_user_pos_items = [set() for _ in range(num_users)]

for user_id, item_id in train_edges.tolist():
    train_user_pos_items[user_id].add(item_id)


# val_items[u] is the single remapped validation item ID for user u.
# A value of -1 means missing, but each retained user should have one item.
val_items = torch.full((num_users,), -1, dtype=torch.long)
val_items[val_edges[:, 0]] = val_edges[:, 1]


# test_items[u] is the single remapped test item ID for user u.
# A value of -1 means missing, but each retained user should have one item.
test_items = torch.full((num_users,), -1, dtype=torch.long)
test_items[test_edges[:, 0]] = test_edges[:, 1]


# trainval_user_pos_items[u] = TRAIN + VAL items for user u. Used as the mask for
# TEST evaluation so baselines match the proposed-method pipeline (ile/popaware),
# which also mask train+val at test time. (Review fix: eval-protocol consistency.)
trainval_user_pos_items = [set(s) for s in train_user_pos_items]
for user_id, item_id in val_edges.tolist():
    trainval_user_pos_items[user_id].add(item_id)


assert len(train_user_pos_items) == num_users
assert (val_items >= 0).sum().item() == num_users
assert (test_items >= 0).sum().item() == num_users

# Build Bipartite Graph

In [ ]:
movie_offset = num_users

user_nodes = train_edges[:, 0]
movie_nodes = train_edges[:, 1] + movie_offset

src = torch.cat([user_nodes, movie_nodes])
dst = torch.cat([movie_nodes, user_nodes])

edge_index_train = torch.stack([src, dst], dim=0)

print("\nGraph")
print(f"edge_index_train shape: {tuple(edge_index_train.shape)}")

# Item degree: computed ONLY from training graph

In [ ]:
item_degree = (
    train_df
    .groupby("movie_idx")
    .size()
    .reindex(range(num_items), fill_value=0)
)

item_degree = torch.tensor(
    item_degree.values,
    dtype=torch.long
)

# Split item to head/middle/tail
Tail   : bottom 50% <br>
Middle : 30% <br>
Head   : top 20% <br>

Encoding: <br>

0 = Tail <br>
1 = Middle <br>
2 = Head <br>

In [ ]:
deg_np = item_degree.numpy()

p50 = np.percentile(deg_np, 50)
p80 = np.percentile(deg_np, 80)

item_popularity_group = torch.zeros(
    num_items,
    dtype=torch.long
)

item_popularity_group[deg_np >= p50] = 1
item_popularity_group[deg_np >= p80] = 2

print("\nPopularity Groups")
print(f"Tail   : {(item_popularity_group == 0).sum().item()}")
print(f"Middle : {(item_popularity_group == 1).sum().item()}")
print(f"Head   : {(item_popularity_group == 2).sum().item()}")



# Statistics

In [ ]:
dataset_statistics = pd.DataFrame({

    "Metric": [
        "Users",
        "Items",
        "Positive interactions",
        "Train interactions",
        "Validation interactions",
        "Test interactions",
        "Average interactions / user",
        "Average interactions / item"
    ],

    "Value": [
        num_users,
        num_items,
        len(df),
        len(train_df),
        len(val_df),
        len(test_df),
        round(len(df) / num_users, 2),
        round(len(df) / num_items, 2)
    ]

})

print("\nDataset Statistics")
print(dataset_statistics)

# =========================================================
# Final output sanity checks
# =========================================================
print("\nSanity Checks")
print(f"number of users                  : {num_users}")
print(f"number of items                  : {num_items}")
print(f"train_edges shape                : {tuple(train_edges.shape)}")
print(f"val_edges shape                  : {tuple(val_edges.shape)}")
print(f"test_edges shape                 : {tuple(test_edges.shape)}")
print(f"edge_index_train shape           : {tuple(edge_index_train.shape)}")
print(f"item_degree shape                : {tuple(item_degree.shape)}")
print(f"item_popularity_group shape      : {tuple(item_popularity_group.shape)}")
print(f"users in train_user_pos_items    : {len(train_user_pos_items)}")
print(f"validation users                 : {(val_items >= 0).sum().item()}")
print(f"test users                       : {(test_items >= 0).sum().item()}")

# =========================================================
# Final objects to hand over
# =========================================================
outputs = {

    "train_edges": train_edges,

    "val_edges": val_edges,

    "test_edges": test_edges,

    "edge_index_train": edge_index_train,

    "num_users": num_users,

    "num_items": num_items,

    "item_degree": item_degree,

    "item_popularity_group": item_popularity_group,

    "train_user_pos_items": train_user_pos_items,

    "val_items": val_items,

    "test_items": test_items,

    "dataset_statistics": dataset_statistics
}

print("\nPreprocessing completed successfully!")

# Evaluation Pipeline Test: MostPopular Baseline

In [ ]:
from pathlib import Path
import sys

# Allow imports from src/ when this notebook is executed from notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.baselines import MostPopular
from src.metrics import evaluate_full_ranking


# MostPopular assigns every user the same training-set item popularity scores.
# It is used here only as a sanity check for the full-ranking evaluation
# pipeline, including training-item masking inside evaluate_full_ranking.
mostpopular = MostPopular().fit(train_edges, num_items)
mostpopular_scores = mostpopular.predict_all(num_users)

mostpopular_results = evaluate_full_ranking(
    scores=mostpopular_scores,
    train_user_pos_items=trainval_user_pos_items,  # train+val mask (review fix)
    test_items=test_items,
    item_group=item_popularity_group,
    item_degree=item_degree,
    k_list=[10, 20],
)

print("\nMostPopular result dictionary")
print(mostpopular_results)

print("\nMostPopular metrics")
for metric_name, metric_value in mostpopular_results.items():
    print(f"{metric_name}: {metric_value:.6f}")

metrics_dir = project_root / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

mostpopular_results_df = pd.DataFrame([mostpopular_results])
mostpopular_results_path = metrics_dir / "mostpopular_results.csv"
mostpopular_results_df.to_csv(mostpopular_results_path, index=False)

print(f"\nSaved MostPopular results to: {mostpopular_results_path}")

# BPR-MF and LightGCN Baselines (Member 3)

Train the two personalized baselines (`BPR-MF` and `LightGCN`), build a full score
matrix `[num_users, num_items]` for each, and evaluate them with the **same**
`evaluate_full_ranking` used for MostPopular. The models never mask training items
themselves — masking happens inside `evaluate_full_ranking`.

Requires the preprocessing cells above to have run (they define `train_edges`,
`edge_index_train`, `num_users`, `num_items`, `train_user_pos_items`, `test_items`,
`item_popularity_group`, `item_degree`, `mostpopular_results`, `metrics_dir`).

In [ ]:
import torch
import pandas as pd

from src import config
from src.train import train_bpr_mf, train_lightgcn
from src.metrics import evaluate_full_ranking

device = config.get_device()
config.set_seed(config.SEED)
print("Device:", device)
print(f"epochs={config.NUM_EPOCHS}  dim={config.EMBEDDING_DIM}  layers={config.NUM_LAYERS}  "
      f"lr={config.LR}  batch={config.BATCH_SIZE}  weight_decay={config.WEIGHT_DECAY}")

In [ ]:
# ---------- BPR-MF ----------
bpr_model, bpr_history = train_bpr_mf(
    train_edges=train_edges,
    num_users=num_users,
    num_items=num_items,
    train_user_pos_items=train_user_pos_items,
    device=device,
)

# Full score matrix [num_users, num_items] -> straight into Member 2's evaluator.
bpr_scores = bpr_model.full_sort_scores().cpu()
assert bpr_scores.shape == (num_users, num_items), bpr_scores.shape

bpr_results = evaluate_full_ranking(
    scores=bpr_scores,
    train_user_pos_items=trainval_user_pos_items,  # train+val mask (review fix)
    test_items=test_items,
    item_group=item_popularity_group,
    item_degree=item_degree,
    k_list=[10, 20],
)

print("\nBPR-MF results")
for name, value in bpr_results.items():
    print(f"{name}: {value:.6f}")

pd.DataFrame([bpr_results]).to_csv(metrics_dir / "bpr_mf_results.csv", index=False)
print("\nSaved:", metrics_dir / "bpr_mf_results.csv")

In [ ]:
# ---------- LightGCN ----------
lightgcn_model, lightgcn_history = train_lightgcn(
    train_edges=train_edges,
    edge_index_train=edge_index_train,
    num_users=num_users,
    num_items=num_items,
    train_user_pos_items=train_user_pos_items,
    device=device,
)

# Full score matrix [num_users, num_items] -> straight into Member 2's evaluator.
lightgcn_scores = lightgcn_model.full_sort_scores(edge_index_train).cpu()
assert lightgcn_scores.shape == (num_users, num_items), lightgcn_scores.shape

lightgcn_results = evaluate_full_ranking(
    scores=lightgcn_scores,
    train_user_pos_items=trainval_user_pos_items,  # train+val mask (review fix)
    test_items=test_items,
    item_group=item_popularity_group,
    item_degree=item_degree,
    k_list=[10, 20],
)

print("\nLightGCN results")
for name, value in lightgcn_results.items():
    print(f"{name}: {value:.6f}")

pd.DataFrame([lightgcn_results]).to_csv(metrics_dir / "lightgcn_results.csv", index=False)
print("\nSaved:", metrics_dir / "lightgcn_results.csv")

In [ ]:
# ---------- Comparison: MostPopular vs BPR-MF vs LightGCN ----------
main_cols = [
    "Model", "Recall@10", "NDCG@10", "Recall@20", "NDCG@20",
    "TailRecall@20", "Coverage@20", "ARP@20",
    "TailExposure@20", "MiddleExposure@20", "HeadExposure@20",
]
comparison = pd.DataFrame([
    {"Model": "MostPopular", **mostpopular_results},
    {"Model": "BPR-MF",      **bpr_results},
    {"Model": "LightGCN",    **lightgcn_results},
])[main_cols]

comparison.to_csv(metrics_dir / "main_results.csv", index=False)
print("Saved comparison to", metrics_dir / "main_results.csv")
comparison.round(4)

In [ ]:
import sys
import torch

print(sys.executable)
print(torch.__file__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

# Proposed Method (ILE + degree-aware CL) & Full Experiments

Phần proposed method (LightGCN + **Item Loss Equalization** và các biến thể **PopAware** kết hợp
degree-aware contrastive learning) được cài trong `src/` — `ile_losses.py`, `ile_training.py`,
`graph_augmentation.py`, `popaware_training.py` — và huấn luyện qua script trên GPU
(`train_final_seeds.py`, `train_all_popaware.py`) với multi-seed {0,1,42}. Kết quả đã lưu ở
`results/` nên notebook này chỉ cần **tổng hợp lại** để dựng bảng và figures, không train lại.

# Results, Figures & Table (tái lập từ kết quả đã lưu)

Cell dưới gom mọi kết quả về `results/results.csv`, sinh **bảng chính (§4.4)** và **6 figures**
vào `figures/`. Chạy được trên CPU trong vài giây.

In [ ]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists() and (project_root.parent / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.collect_results import main as collect_results
from src.make_table import main as make_table
from src import plots

# 1) Consolidate all model results into results/results.csv
collect_results()

# 2) Main results table (Markdown + LaTeX) -> results/main_results_table.{md,tex}
make_table()

# 3) Figures 1-6 -> figures/
item_degree = torch.load(project_root / 'preprocess_data' / 'item_degree.pt', weights_only=True)
item_group  = torch.load(project_root / 'preprocess_data' / 'item_popularity_group.pt', weights_only=True)
plots.plot_item_degree_distribution(item_degree)          # Fig 1: long-tail (popularity bias)
plots.plot_popularity_group_distribution(item_group)      # Fig 2: 20/30/50 groups
plots.plot_recall_vs_tail_recall()                        # Fig 3: accuracy-diversity trade-off
plots.plot_coverage_by_model()                            # Fig 4: catalog coverage
plots.plot_exposure_by_group()                            # Fig 5: head/middle/tail exposure
hist_dir = project_root / 'results' / 'popaware'
plots.plot_training_loss_from_history({                   # Fig 6: training loss curve
    'LightGCN (baseline)': hist_dir / 'history_final_baseline_L2_s42.csv',
    'PopAware-BEST':       hist_dir / 'history_final_best_L2_s42.csv',
    'PopAware-high-tail':  hist_dir / 'history_final_hightail_L2_s42.csv',
}, loss_column='bpr')
print('Done: results/results.csv, results/main_results_table.md, figures/fig1..6')

# Reproducibility notes

- **Seed**: cố định qua `src.config.set_seed`; các dòng chính (LightGCN, PopAware) chạy 3 seed
  {0, 1, 42} và báo cáo mean ± std.
- **Config chung**: mọi hyperparameter ở `src/config.py` (embedding_dim, num_layers, lr,
  batch_size, λ_ILE, K...). Không hardcode ở nơi khác.
- **Evaluation**: dùng chung `src.metrics.evaluate_full_ranking`; TEST mask = train+val cho
  MỌI model (baseline lẫn proposed) — xem `NOTIFICATION_eval_mask_fix.md`.
- **Thứ tự chạy**: Restart & Run All. Data (cell đầu) → baselines → cell tổng hợp cuối. Các
  run proposed nặng đã thực hiện offline; kết quả lưu sẵn trong `results/`.